# Performance Analysis — ollama Load Test

This notebook loads `metrics.csv` produced by `load_test.py` and visualises
latency, throughput, and time-to-first-token across concurrency levels,
prompt types, and configuration variants.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os

CSV_PATH = os.path.join(os.path.dirname(os.getcwd()), "perf", "metrics.csv")
if not os.path.exists(CSV_PATH):
    CSV_PATH = "metrics.csv"

df = pd.read_csv(CSV_PATH)
for col in ["ttft_ms", "tpot", "latency_p50", "latency_p95", "latency_p99"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df.head(10)

## 1. Latency Distribution by Concurrency Level

Grouped bar chart comparing P50 / P95 / P99 latency at each concurrency
level.  Higher concurrency increases contention on the model, which should
push tail latencies up even if median stays relatively stable.

In [ ]:
agg = df.groupby("concurrency")[["latency_p50", "latency_p95", "latency_p99"]].mean()

fig, ax = plt.subplots(figsize=(9, 5))
x = np.arange(len(agg))
w = 0.25
ax.bar(x - w, agg["latency_p50"], w, label="P50")
ax.bar(x,     agg["latency_p95"], w, label="P95")
ax.bar(x + w, agg["latency_p99"], w, label="P99")
ax.set_xticks(x)
ax.set_xticklabels(agg.index)
ax.set_xlabel("Concurrency")
ax.set_ylabel("Latency (ms)")
ax.set_title("Latency Percentiles by Concurrency")
ax.legend()
plt.tight_layout()
plt.savefig("plot_latency_by_concurrency.png", dpi=150)
plt.show()

## 2. Time-to-First-Token vs Prompt Type

TTFT is critical for user-perceived responsiveness.  Long prompts require
more prefill computation, so we expect higher TTFT.  Each dot is one
configuration row; the box shows spread across concurrency and cache states.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
for ptype, grp in df.groupby("prompt_type"):
    ax.scatter(grp["concurrency"], grp["ttft_ms"], label=ptype, alpha=0.7, s=60)

ax.set_xlabel("Concurrency")
ax.set_ylabel("TTFT (ms)")
ax.set_title("Time-to-First-Token by Prompt Type")
ax.legend()
plt.tight_layout()
plt.savefig("plot_ttft_by_prompt.png", dpi=150)
plt.show()

## 3. Throughput vs Concurrency

Tokens-per-second (TPS) should ideally increase with concurrency if the
server can batch, but on CPU-only setups ollama serialises requests, so
per-request throughput often stays flat or drops slightly.

In [ ]:
tps_agg = df.groupby("concurrency")["tpot"].mean()

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(tps_agg.index, tps_agg.values, marker="o", linewidth=2)
ax.set_xlabel("Concurrency")
ax.set_ylabel("Avg Tokens / sec")
ax.set_title("Throughput vs Concurrency")
ax.set_xticks(tps_agg.index)
plt.tight_layout()
plt.savefig("plot_throughput.png", dpi=150)
plt.show()

## 4. Cache Impact on Latency

Ollama keeps the model weights in memory after the first request ("warm").
The first cold request pays a loading penalty.  Subsequent warm requests
should be faster.

In [ ]:
cache_agg = df.groupby("cache")[["latency_p50", "latency_p95"]].mean()

fig, ax = plt.subplots(figsize=(6, 4))
cache_agg.plot(kind="bar", ax=ax)
ax.set_ylabel("Latency (ms)")
ax.set_title("Cold vs Warm Cache Latency")
ax.set_xticklabels(cache_agg.index, rotation=0)
plt.tight_layout()
plt.savefig("plot_cache_impact.png", dpi=150)
plt.show()

## 5. Stop-Sequence Impact

Adding stop sequences can terminate generation early, reducing latency
for prompts where the stop pattern appears quickly.

In [ ]:
stop_agg = df.groupby("stop_seq")[["latency_p50", "tpot"]].mean()

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
stop_agg["latency_p50"].plot(kind="bar", ax=axes[0])
axes[0].set_title("P50 Latency by Stop-Seq")
axes[0].set_ylabel("ms")
axes[0].set_xticklabels(["No Stop", "With Stop"], rotation=0)

stop_agg["tpot"].plot(kind="bar", ax=axes[1])
axes[1].set_title("Throughput by Stop-Seq")
axes[1].set_ylabel("tok/s")
axes[1].set_xticklabels(["No Stop", "With Stop"], rotation=0)

plt.tight_layout()
plt.savefig("plot_stop_seq.png", dpi=150)
plt.show()

## Key Observations

- **Concurrency**: On CPU, ollama serialises inference so higher concurrency
  increases queue wait, raising P95/P99 while P50 stays roughly flat.
- **TTFT**: Long prompts consistently show higher time-to-first-token due to
  larger prefill computation.
- **Cache**: The first (cold) batch pays a significant model-loading penalty.
  Subsequent warm batches are noticeably faster.
- **Stop sequences**: When effective, stop sequences truncate generation
  early, reducing total latency without affecting per-token speed.